In [0]:
# Notebook 2: Limpieza de datos (capa Silver)

df = spark.table("workspace.default.retail_raw")

print(f"Filas en raw: {df.count()}")
print(f"Columnas: {df.columns}")

In [0]:
# Renombrar columnas a snake_case (alineado con el modelo dimensional)
df = (df
      .withColumnRenamed("Transaction ID", "transaction_id")
      .withColumnRenamed("Customer ID", "customer_id")
      .withColumnRenamed("Category", "category")
      .withColumnRenamed("Item", "item")
      .withColumnRenamed("Price Per Unit", "price_per_unit")
      .withColumnRenamed("Quantity", "quantity")
      .withColumnRenamed("Total Spent", "total_spent")
      .withColumnRenamed("Payment Method", "payment_method")
      .withColumnRenamed("Location", "location")
      .withColumnRenamed("Transaction Date", "transaction_date")
      .withColumnRenamed("Discount Applied", "discount_applied"))

print("Nombres normalizados:")
for c in df.columns:
    print(f"  - {c}")

In [0]:
from pyspark.sql.functions import col

# Contar nulos por columna
total = df.count()
print(f"Total de filas: {total}\n")
print("Nulos por columna:")
for c in df.columns:
    nulos = df.filter(col(c).isNull()).count()
    if nulos > 0:
        pct = nulos / total * 100
        print(f"  {c}: {nulos} ({pct:.1f}%)")

In [0]:
# Partimos del DataFrame y aplicamos las reglas en orden
df_clean = df.filter(col("item").isNotNull())
print(f"Filas tras eliminar items nulos: {df_clean.count()}")

In [0]:
from pyspark.sql.functions import col, when, round

# Completar price_per_unit si falta
df_clean = df_clean.withColumn(
    "price_per_unit",
    when(
        col("price_per_unit").isNull(),
        round(col("total_spent") / col("quantity"), 2)
    ).otherwise(col("price_per_unit"))
)

# Completar quantity si falta
df_clean = df_clean.withColumn(
    "quantity",
    when(
        col("quantity").isNull(),
        round(col("total_spent") / col("price_per_unit")).cast("int")
    ).otherwise(col("quantity"))
)


# Verificar cuántos nulos quedan
nulos_price = df_clean.filter(col("price_per_unit").isNull()).count()
nulos_qty = df_clean.filter(col("quantity").isNull()).count()
print(f"Nulos restantes en price_per_unit: {nulos_price}")
print(f"Nulos restantes en quantity: {nulos_qty}")

In [0]:
# Las filas que aún tengan price_per_unit, quantity o total_spent nulos se descartan
df_clean = df_clean.filter(
    col("price_per_unit").isNotNull() &
    col("quantity").isNotNull()
)
print(f"Filas tras eliminar irrecuperables: {df_clean.count()}")

In [0]:
# Total Spent siempre se recalcula como price × quantity
df_clean = df_clean.withColumn(
    "total_spent",
    round(col("price_per_unit") * col("quantity"), 2)
)

# Mostrar 5 filas para verificar
df_clean.select("transaction_id", "price_per_unit", "quantity", "total_spent").show(5)

In [0]:
from pyspark.sql.functions import lit

# Convertir el campo a booleano y reemplazar nulos por False
df_clean = df_clean.withColumn(
    "discount_applied",
    when(col("discount_applied") == "True", lit(True))
    .when(col("discount_applied") == "False", lit(False))
    .otherwise(lit(False))  # Los nulos se asumen False
)

# Verificar la distribución
df_clean.groupBy("discount_applied").count().show()

In [0]:
# Solo se aceptan los valores válidos definidos en el modelo
df_clean = df_clean.filter(
    col("payment_method").isin("Cash", "Credit Card", "Digital Wallet") &
    col("location").isin("Online", "In-store")
)
print(f"Filas tras validar dominios: {df_clean.count()}")

In [0]:
from pyspark.sql.functions import to_date

df_clean = df_clean.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "yyyy-MM-dd")
)

# Verificar el tipo
df_clean.printSchema()

In [0]:
filas_finales = df_clean.count()
print(f"\n=== RESUMEN DE LIMPIEZA ===")
print(f"Filas originales: 12575")
print(f"Filas finales (limpias): {filas_finales}")
print(f"Filas eliminadas: {12575 - filas_finales}")
print(f"Porcentaje retenido: {filas_finales/12575*100:.1f}%")

(df_clean.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")
 .saveAsTable("workspace.default.retail_clean"))

print("\nTabla 'retail_clean' creada exitosamente.")